In [0]:
# %sql
# -- 1. Aseguramos que la tabla no tenga basura previa
# DROP TABLE IF EXISTS workspace.products.silver_products;

In [0]:
%sql
DESCRIBE TABLE workspace.products.catalog;

In [0]:
%sql
DESCRIBE TABLE workspace.products.bronze_scraped_products;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS total_products,
    COUNT(DISTINCT retailer) AS total_retailers,
    COUNT(DISTINCT brand) AS total_brands,
    COUNT(DISTINCT sku) AS total_skus,
    MIN(scraped_at) AS first_scrape,
    MAX(scraped_at) AS last_scrape
FROM
    workspace.products.silver_products

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS workspace.products.silver_products (
    product_id STRING,
    retailer STRING,
    brand STRING,
    sku STRING,
    name STRING,
    list_price DOUBLE,
    cash_price DOUBLE,
    effective_price DOUBLE,
    scraped_at TIMESTAMP,
    scraped_date DATE,
    discount_pct DOUBLE,
    installments_json STRING,
    specification_json STRING,
    description STRING,
    rating DOUBLE,
    stock STRING,
    is_available BOOLEAN,
    is_valid_name BOOLEAN,
    is_valid_price BOOLEAN,
    discount_applied_str STRING,
    discount_applied_int DOUBLE,
    has_installments BOOLEAN,
    has_description BOOLEAN
) 
USING DELTA 
PARTITIONED BY (scraped_date, retailer);

ALTER TABLE workspace.products.silver_products SET
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
DESCRIBE TABLE workspace.products.silver_products;

In [0]:
%sql
MERGE INTO workspace.products.silver_products AS target
USING (
  WITH cleaned_prices AS (
    SELECT 
      b.product_id,
      b.retailer,
      b.scraped_at,
      get_json_object(b.raw_data, '$.sku') as sku,
      COALESCE(
        get_json_object(b.raw_data, '$.name'), 
        get_json_object(b.raw_data, '$.title')
      ) as consolidated_name,
      COALESCE(
        TRY_CAST(get_json_object(b.raw_data, '$.list_price') AS DOUBLE),
        TRY_CAST(REGEXP_REPLACE(get_json_object(b.raw_data, '$.list_price'), '[\\$.]', '') AS DOUBLE)
      ) AS clean_list,
      COALESCE(
        TRY_CAST(get_json_object(b.raw_data, '$.cash_price') AS DOUBLE),
        TRY_CAST(REGEXP_REPLACE(get_json_object(b.raw_data, '$.cash_price'), '[\\$.]', '') AS DOUBLE)
      ) AS clean_cash,
      get_json_object(b.raw_data, '$.rating') as rating,
      get_json_object(b.raw_data, '$.installments') as installments,
      -- Corrige el campo specifications para extraer correctamente el JSON
      get_json_object(b.raw_data, '$.specifications') as specifications,
      get_json_object(b.raw_data, '$.description') as description,
      get_json_object(b.raw_data, '$.stock') AS is_in_stock,
      c.brand AS catalog_brand
    FROM workspace.products.bronze_scraped_products b
    INNER JOIN workspace.products.catalog c 
      ON b.product_id = c.product_id AND b.retailer = c.retailer
  )
  ,
  intermediate_data AS (
    SELECT 
      product_id,
      retailer,
      catalog_brand AS brand,
      sku,
      consolidated_name AS name,
      clean_list AS list_price,
      clean_cash AS cash_price,
      COALESCE(clean_cash, clean_list) AS effective_price,
      scraped_at,
      CAST(scraped_at AS DATE) AS scraped_date,
      ROUND(COALESCE((1 - (clean_cash / NULLIF(clean_list, 0))) * 100, 0.0), 2) AS clean_discount,
      -- ROUND((1 - (clean_cash / clean_list)) * 100, 2) AS discount_pct,
      installments AS installments_json,
      -- Cambia specification_json por specifications para que no sea null
      specifications AS specification_json,
      description,
      CASE 
        WHEN rating IS NULL OR rating = '{}' OR rating = '' THEN NULL 
        ELSE TRY_CAST(get_json_object(rating, '$.ratingValue') AS DOUBLE)
      END AS rating_value,
      is_in_stock AS stock,
      (clean_list IS NOT NULL OR clean_cash IS NOT NULL) AS is_available,
      (consolidated_name IS NOT NULL AND consolidated_name != '') AS is_valid_name,
      CASE 
        WHEN clean_list IS NULL AND clean_cash IS NULL THEN FALSE
        WHEN clean_list > 20000000 OR clean_list < 0 THEN FALSE
        WHEN clean_cash > 20000000 OR clean_cash < 0 THEN FALSE
        WHEN ((ABS(clean_list - clean_cash) / NULLIF(LEAST(clean_list, clean_cash), 0)) * 100) > 500 THEN FALSE
        ELSE TRUE 
      END AS is_valid_price,
      CASE 
        WHEN TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
          OR TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
          OR TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN '0'
        ELSE CONCAT(ROUND((1 - (TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2), '%')
      END AS discount_applied_str,
      CASE 
        WHEN TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
          OR TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
          OR TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN 0.0
        ELSE ROUND((1 - (TRY_CAST(REGEXP_REPLACE(cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2)
      END AS discount_applied_int,
      CASE 
        WHEN installments IS NULL OR installments = '{}' OR installments = '' THEN FALSE 
        ELSE TRUE 
      END AS has_installments,
      CASE 
        WHEN description IS NULL OR description = '' THEN FALSE 
        ELSE TRUE 
      END AS has_description,
      ROW_NUMBER() OVER (PARTITION BY product_id, retailer, scraped_at ORDER BY product_id) AS rn
    FROM cleaned_prices
  )
  SELECT 
    product_id,
    retailer,
    brand,
    sku,
    name,
    list_price,
    cash_price,
    effective_price,
    scraped_at,
    scraped_date,
    clean_discount AS discount_pct,
    installments_json,
    specification_json,
    description,
    rating_value AS rating,
    stock,
    is_available,
    is_valid_name,
    is_valid_price,
    CASE WHEN clean_discount = 0.0 THEN '0' ELSE CONCAT(clean_discount, '%') END AS discount_applied_str, -- Reutilizado sin re-calcular Regex
    clean_discount AS discount_applied_int,
    has_installments,
    has_description
  FROM intermediate_data
  WHERE rn = 1
) AS source
ON target.product_id = source.product_id 
   AND target.retailer = source.retailer 
   AND target.scraped_at = source.scraped_at
-- Si ya existe esa combinación exacta, podrías actualizar (opcional)
WHEN MATCHED THEN
  UPDATE SET 
    target.list_price = source.list_price,
    target.cash_price = source.cash_price,
    target.effective_price = source.effective_price,
    target.discount_pct = source.discount_pct,
    target.installments_json = source.installments_json,
    target.specification_json = source.specification_json,
    target.description = source.description,
    target.rating = source.rating,
    target.stock = source.stock,
    target.is_available = source.is_available,
    target.is_valid_name = source.is_valid_name,
    target.is_valid_price = source.is_valid_price,
    target.discount_applied_str = source.discount_applied_str,
    target.discount_applied_int = source.discount_applied_int,
    target.has_installments = source.has_installments,
    target.has_description = source.has_description
-- Si no existe, es un dato nuevo del scraping de hoy y se inserta
WHEN NOT MATCHED THEN
  INSERT (
    product_id, retailer, brand, sku, name, list_price, cash_price, effective_price, scraped_at, scraped_date, discount_pct, 
    installments_json, specification_json, description, rating, stock,
    is_available, is_valid_name, is_valid_price, discount_applied_str, discount_applied_int, 
    has_installments, has_description
  )
  VALUES (
    source.product_id, 
    source.retailer, 
    source.brand, 
    source.sku,
    source.name, 
    source.list_price, 
    source.cash_price,
    source.effective_price,
    source.scraped_at, 
    source.scraped_date,
    source.discount_pct,
    source.installments_json,
    source.specification_json,
    source.description,
    source.rating,
    source.stock,
    source.is_available,
    source.is_valid_name,
    source.is_valid_price,
    source.discount_applied_str,
    source.discount_applied_int,
    source.has_installments,
    source.has_description
  )

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS total_products,
    COUNT(DISTINCT retailer) AS total_retailers,
    COUNT(DISTINCT brand) AS total_brands,
    COUNT(DISTINCT sku) AS total_skus,
    MIN(scraped_at) AS first_scrape,
    MAX(scraped_at) AS last_scrape
FROM
    workspace.products.silver_products

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    MAX(scraped_at) AS max_scraped_at,
    COUNT(DISTINCT product_id, retailer, brand) AS total_products
FROM
    workspace.products.silver_products
;

## Validaciones de Calidad de Datos Silver

Esta sección valida la calidad de los datos transformados en la tabla `silver_products` para identificar:
- Registros con campos clave faltantes
- Precios fuera de rango o inconsistentes
- Descuentos ilógicos
- Duplicados no esperados
- Inconsistencias de datos

In [0]:
%sql
-- Validación de campos obligatorios que nunca deben ser NULL o vacíos
SELECT 
  'Product ID vacío o NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE product_id IS NULL OR product_id = '' OR TRIM(product_id) = ''

UNION ALL

SELECT 
  'Retailer vacío o NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE retailer IS NULL OR retailer = '' OR TRIM(retailer) = ''

UNION ALL

SELECT 
  'Brand vacío o NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE brand IS NULL OR brand = '' OR TRIM(brand) = ''

UNION ALL

SELECT 
  'SKU vacío o NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE sku IS NULL OR sku = '' OR TRIM(sku) = ''

UNION ALL

SELECT 
  'Nombre vacío o NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE name IS NULL OR name = '' OR TRIM(name) = ''

UNION ALL

SELECT 
  'Ambos precios NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE list_price IS NULL AND cash_price IS NULL

ORDER BY invalid_count DESC;

In [0]:
%sql
-- Validación de rangos de precios y consistencia
SELECT 
  'Precio lista negativo' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE list_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE list_price < 0

UNION ALL

SELECT 
  'Precio efectivo negativo' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE cash_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE cash_price < 0

UNION ALL

SELECT 
  'Precio lista sospechosamente alto (>10M)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE list_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE list_price > 10000000

UNION ALL

SELECT 
  'Precio efectivo sospechosamente alto (>10M)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE cash_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE cash_price > 10000000

UNION ALL

SELECT 
  'Precio efectivo mayor que lista' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE list_price IS NOT NULL AND cash_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE cash_price > list_price
  AND list_price IS NOT NULL 
  AND cash_price IS NOT NULL

UNION ALL

SELECT 
  'Precio sospechosamente bajo (<100)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE list_price IS NOT NULL OR cash_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE (list_price > 0 AND list_price < 100) OR (cash_price > 0 AND cash_price < 100)

ORDER BY invalid_count DESC;

In [0]:
%sql
-- Validación de descuentos y consistencia con precios
SELECT 
  'Descuento mayor al 100%' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE discount_pct IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE discount_pct > 100

UNION ALL

SELECT 
  'Descuento negativo (precio efectivo > lista)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE discount_pct IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE discount_pct < 0

UNION ALL

SELECT 
  'Descuento sospechosamente alto (>80%)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE discount_pct IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE discount_pct > 80 AND discount_pct <= 100

UNION ALL

SELECT 
  'Descuento inconsistente con precios' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE discount_pct IS NOT NULL AND list_price IS NOT NULL AND cash_price IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE list_price IS NOT NULL 
  AND cash_price IS NOT NULL
  AND discount_pct IS NOT NULL
  AND ABS(discount_pct - ROUND((1 - (cash_price / list_price)) * 100, 2)) > 0.5

ORDER BY invalid_count DESC;

In [0]:
%sql
-- Validación de ratings
SELECT 
  'Rating fuera de rango (debe ser 0-5)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE rating IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE rating IS NOT NULL AND (rating < 0 OR rating > 5)

UNION ALL

SELECT 
  'Rating NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE rating IS NULL

UNION ALL

SELECT 
  'Rating = 0 (posiblemente sin reviews)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products WHERE rating IS NOT NULL), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE rating = 0

ORDER BY invalid_count DESC;

In [0]:
%sql
-- Validación de fechas de scraping
SELECT 
  'Fecha de scraping en el futuro' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE scraped_date > CURRENT_DATE()

UNION ALL

SELECT 
  'Fecha de scraping muy antigua (>1 año)' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE scraped_date < DATE_SUB(CURRENT_DATE(), 365)

UNION ALL

SELECT 
  'scraped_at y scraped_date inconsistentes' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE CAST(scraped_at AS DATE) != scraped_date

UNION ALL

SELECT 
  'scraped_at NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE scraped_at IS NULL

UNION ALL

SELECT 
  'scraped_date NULL' AS validation_type,
  COUNT(*) AS invalid_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM workspace.products.silver_products
WHERE scraped_date IS NULL

ORDER BY invalid_count DESC;

In [0]:
%sql
-- Validación de duplicados por clave primaria natural
WITH duplicate_check AS (
  SELECT 
    product_id,
    retailer,
    scraped_at,
    COUNT(*) AS occurrence_count
  FROM workspace.products.silver_products
  GROUP BY product_id, retailer, scraped_at
  HAVING COUNT(*) > 1
)
SELECT 
  'Duplicados por (product_id, retailer, scraped_at)' AS validation_type,
  COUNT(*) AS duplicate_groups,
  SUM(occurrence_count) AS total_duplicate_records,
  ROUND(SUM(occurrence_count) * 100.0 / (SELECT COUNT(*) FROM workspace.products.silver_products), 2) AS pct_of_total
FROM duplicate_check;

In [0]:
%sql
-- Resumen de calidad de datos por retailer
SELECT 
  retailer,
  COUNT(*) AS total_products,
  SUM(CASE WHEN NOT is_valid_name THEN 1 ELSE 0 END) AS invalid_names,
  SUM(CASE WHEN NOT is_valid_price THEN 1 ELSE 0 END) AS invalid_prices,
  SUM(CASE WHEN NOT is_available THEN 1 ELSE 0 END) AS unavailable_products,
  SUM(CASE WHEN name IS NULL OR name = '' THEN 1 ELSE 0 END) AS missing_names,
  SUM(CASE WHEN list_price IS NULL AND cash_price IS NULL THEN 1 ELSE 0 END) AS missing_all_prices,
  SUM(CASE WHEN rating = 0 OR rating IS NULL THEN 1 ELSE 0 END) AS no_rating,
  SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END) AS missing_description,
  ROUND(SUM(CASE WHEN is_valid_name AND is_valid_price THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS quality_score_pct
FROM workspace.products.silver_products
GROUP BY retailer
ORDER BY quality_score_pct DESC;

In [0]:
%sql
-- Top 20 productos con más problemas de calidad de datos
SELECT 
  product_id,
  retailer,
  brand,
  name,
  scraped_date,
  CASE WHEN product_id IS NULL OR product_id = '' THEN 1 ELSE 0 END +
  CASE WHEN name IS NULL OR name = '' THEN 1 ELSE 0 END +
  CASE WHEN list_price IS NULL AND cash_price IS NULL THEN 1 ELSE 0 END +
  CASE WHEN NOT is_valid_price THEN 1 ELSE 0 END +
  CASE WHEN NOT is_valid_name THEN 1 ELSE 0 END +
  CASE WHEN discount_pct < 0 OR discount_pct > 100 THEN 1 ELSE 0 END +
  CASE WHEN rating < 0 OR rating > 5 THEN 1 ELSE 0 END AS quality_issues_count,
  CONCAT_WS(', ',
    CASE WHEN product_id IS NULL OR product_id = '' THEN 'product_id vacío' END,
    CASE WHEN name IS NULL OR name = '' THEN 'nombre vacío' END,
    CASE WHEN list_price IS NULL AND cash_price IS NULL THEN 'sin precios' END,
    CASE WHEN NOT is_valid_price THEN 'precio inválido' END,
    CASE WHEN NOT is_valid_name THEN 'nombre inválido' END,
    CASE WHEN discount_pct < 0 OR discount_pct > 100 THEN 'descuento fuera de rango' END,
    CASE WHEN rating < 0 OR rating > 5 THEN 'rating fuera de rango' END
  ) AS issues_description
FROM workspace.products.silver_products
WHERE scraped_date >= CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY quality_issues_count DESC, product_id
LIMIT 20;